<a href="https://colab.research.google.com/github/lasigeBioTM/data-text-processing-notebooks/blob/main/notebooks/02-data-retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unix Shell Tutorial: Biomedical Data Retrieval

This is the **second tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to access biomedical databases provided by the [European Bioinformatics Institute (EBI)](https://www.ebi.ac.uk/) by using their [web services](https://www.ebi.ac.uk/services) for automated data and text retrieval.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 1: Web Identifiers

The input argument(s) of our retrieval task is the chemical compound(s) of which we want to retrieve more information. For the sake of simplicity, we will start by assuming that the user knows the ChEBI identifier(s), i.e. the script does not have to search by the name of the compounds.
So, the first step is to automatically retrieve all proteins associated to the given input chemical compound, that in our example is caffeine [CHEBI:27732](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:27732).

### Web Identifiers Overview

The new ChEBI 2.0 web interface is currently missing detailed cross-references on entry pages (e.g., UniProt links for caffeine). However, the relation between caffeine and proteins can now be found in Human Metabolome Database (HMDB) [https://hmdb.ca/metabolites/HMDB0001847/metabolite_protein_links](https://hmdb.ca/metabolites/HMDB0001847/metabolite_protein_links) or retrieved programmatically using EBI web services.

To retrieve the data, we can use the following URLs:

1. [https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=csv](https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=csv)
2. [https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=tsv](https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=tsv)
3. [https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=json](https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=json)
4. [https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=xml](https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=50&format=xml)

for downloading a CSV, TSV, JSON, or XML file, respectively, with the first 50 associated proteins.

We should note that the only difference between the URLs is the value (`csv`, `tsv`, `json`, and `xml`) assigned to the `format` parameter after the ampersand character (`&`), which means that this value can be used as an argument to select the type of file.

We should note that the ChEBI identifier (27732) is easily observable in the URL. Try to replace `27732` by `17245` in that URL, for example:

- [https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/17245/xref/UniProtKB?size=50&format=xml](https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/17245/xref/UniProtKB?size=50&format=xml)

Now we can use this new URL in the internet browser, and check what happens. If we did it correctly, our browser downloaded (or displayed) an XML file with protein references, since `17245` is the ChEBI identifier of a popular chemical compound in life systems, the carbon monoxide.

In this case, we are using a fully RESTful web service, where the data path is pretty modular and self-explanatory. The path is clearly composed of:

- the name of the database (`chebi`);
- the entry resource (`entry`);
- the specific identifier (`27732`);
- the cross-reference resource (`xref/UniProtKB`);
- and a list of parameters and their value (arguments) after the question mark character (?).

The order of the parameters in the URL is normally not relevant. They are separated by the ampersand character (`&`) and the equals character (`=`) is used to assign a value to each parameter (`argument`). This modular structure of these URLs allows us to use them as data pipelines to fill our local files with data, like pipelines that transport oil or gas from one container to another.


### Single and Double Quotes

To construct the URL for a given ChEBI identifier, let us first understand the difference between single quotes and double quotes in a string (sequence of characters). We can create a script file named `getproteins.sh`:

In [28]:
%%bash
cat > getproteins.sh << 'EOF'
echo 'The input: $1'
echo "The input: $1"
EOF

The command line tool `echo` displays the string received as argument.

Do not forget to save it in our working directory:

In [29]:
%%bash
cat getproteins.sh

echo 'The input: $1'
echo "The input: $1"


And add the right permissions with `chmod` as we did previously with our first script:

In [30]:
%%bash
chmod u+x getproteins.sh

**Expected Output:** No output (command executed successfully).

Now to execute the script we will only need to type:

In [31]:
%%bash
./getproteins.sh

The input: $1
The input: 


This means that when using single quotes, the string is interpreted literally as it is, whereas the string within double quotes is analyzed, and if there is a special character, such as the dollar sign (`$`), the script translates it to what it represents. In this case, `$1` represents the first input argument. Since no
argument was given, the double quotes displays nothing.

To execute the script with an argument, we can type:

In [32]:
%%bash
./getproteins.sh 27732

The input: $1
The input: 27732


We can check now that when using double quotes, `$1` is translated to the string given as argument.

Now we can update our script file named `getproteins.sh` to contain only the following line:

```bash
echo "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=100&format=csv"
```


In [33]:
%%bash
cat > getproteins.sh << 'EOF'
echo "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"
EOF

### Comments

Instead of removing the previous lines, we can transform them in comments
by adding the hash character (`#`) to the beginning of the line:

```bash
#echo 'The input: $1'
#echo "The input: $1"
echo "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"
```

Commented lines are ignored by the computer during script execution, so this script will perform the same actions as the previous one with just one line of code.

Now, we can execute the script giving the ChEBI identifier as argument:

In [34]:
%%bash
./getproteins.sh 27773

https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27773/xref/UniProtKB?size=100&format=csv


The output on our terminal should be the link that returns the CSV file containing the proteins associated with caffeine.

The next step will use this link to retrieve the data.

## Step 2: Data Retrieval

After having the link, we need a web retrieval tool that works like our internet browser, i.e., receives as input a URL for programmatic access and retrieves its contents from the internet. We will use Client Uniform Resource Locator
(cURL), which is available as a command line tool, and allows us to download
the result of opening a URL directly into a file.

> **Note**: The `curl` command works the same way in your local terminal. In this notebook environment, the command retrieves data from the EBI web service.

For example, to display in our screen the list of proteins related to caffeine,
we just need to add the respective URL as input argument:

In [35]:
%%bash
curl 'https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=100&format=csv'

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"27732","CHEBI:27732","chebi","ICS1_CAMIR","Q2HXL9","uniprot"
"27732","CHEBI:27732","chebi","CKCS_CAMSB","A0A6C0WW38","uniprot"
"27732","CHEBI:27732","chebi","TCS1D_CAMTA","A0A0S2PMA8","uniprot"
"27732","CHEBI:27732","chebi","CHIB1_ASPFM","Q873X9","uniprot"
"27732","CHEBI:27732","chebi","RYR1_PIG","P16960","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFAR","Q9AVK1","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFCA","A0A096VHZ1","uniprot"
"27732","CHEBI:27732","chebi","TCS2_CAMSI","Q68CM3","uniprot"
"27732","CHEBI:27732","chebi","TCS3_CAMSI","A0A0S2PM82","uniprot"
"27732","CHEBI:27732","chebi","CDHC_PSEU3","D7REY5","uniprot"
"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","CDHB_PSEU3","D7REY4","uniprot"
"27732","CHEBI:27732","chebi","TCS5_CAMSI","L0BUM3","uniprot"
"27732","CHEBI:27732","chebi","NDMA_PSEPU","H9N289","uniprot"
"27732","CHEBI:27732","chebi","PCS1_CAMPL","Q2HXI6","u

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1890    0  1890    0     0   3283      0 --:--:-- --:--:-- --:--:--  3281


The output on our terminal is the long list of proteins related to caffeine (ChEBI:27732).

An alternative to `curl` is the command `wget` (not available in this platform), which also receives a URL as argument. By default, `wget` writes the contents to a file instead of displaying it on the screen. The equivalent command is to add the `-O-` option to specify where the contents should be placed.

```bash
wget -O- 'https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/27732/xref/UniProtKB?size=100&format=csv'
```

Instead of using a fixed URL, we can update the script to contain only the following line:

```bash
curl "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"
```

In [36]:
%%bash
cat > getproteins.sh << 'EOF'
curl "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=99&format=csv"
EOF

We should note that now we are using double quotes, since we replaced the
caffeine identifier by `$1`.

Now to execute the script we only need to provide a ChEBI identifier as input argument:

In [37]:
%%bash
./getproteins.sh 27732

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"27732","CHEBI:27732","chebi","ICS1_CAMIR","Q2HXL9","uniprot"
"27732","CHEBI:27732","chebi","CKCS_CAMSB","A0A6C0WW38","uniprot"
"27732","CHEBI:27732","chebi","TCS1D_CAMTA","A0A0S2PMA8","uniprot"
"27732","CHEBI:27732","chebi","CHIB1_ASPFM","Q873X9","uniprot"
"27732","CHEBI:27732","chebi","RYR1_PIG","P16960","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFAR","Q9AVK1","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFCA","A0A096VHZ1","uniprot"
"27732","CHEBI:27732","chebi","TCS2_CAMSI","Q68CM3","uniprot"
"27732","CHEBI:27732","chebi","TCS3_CAMSI","A0A0S2PM82","uniprot"
"27732","CHEBI:27732","chebi","CDHC_PSEU3","D7REY5","uniprot"
"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","CDHB_PSEU3","D7REY4","uniprot"
"27732","CHEBI:27732","chebi","TCS5_CAMSI","L0BUM3","uniprot"
"27732","CHEBI:27732","chebi","NDMA_PSEPU","H9N289","uniprot"
"27732","CHEBI:27732","chebi","PCS1_CAMPL","Q2HXI6","u

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1890    0  1890    0     0   4412      0 --:--:-- --:--:-- --:--:--  4405


The output on our terminal is the long list of proteins related to ChEBI:27773.

Or, if we want the proteins related to carbon monoxide, we only need to
replace the argument:

In [38]:
%%bash
./getproteins.sh 17245

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"17245","CHEBI:17245","chebi","A0A0H2M9I4_VARPD","A0A0H2M9I4","uniprot"
"17245","CHEBI:17245","chebi","A0ABM6JKM8_9GAMM","A0ABM6JKM8","uniprot"
"17245","CHEBI:17245","chebi","N9GGR0_ACIHA","N9GGR0","uniprot"
"17245","CHEBI:17245","chebi","A0A6V6Y4L7_9FIRM","A0A6V6Y4L7","uniprot"
"17245","CHEBI:17245","chebi","A0ABS7SSJ5_9BURK","A0ABS7SSJ5","uniprot"
"17245","CHEBI:17245","chebi","A0ABV5WD39_9BACI","A0ABV5WD39","uniprot"
"17245","CHEBI:17245","chebi","A0A419T058_9FIRM","A0A419T058","uniprot"
"17245","CHEBI:17245","chebi","A0ABM6V057_9GAMM","A0ABM6V057","uniprot"
"17245","CHEBI:17245","chebi","A0A917KCP1_9BACL","A0A917KCP1","uniprot"
"17245","CHEBI:17245","chebi","F1TGS5_9FIRM","F1TGS5","uniprot"
"17245","CHEBI:17245","chebi","A0A7X5J8J1_9HYPH","A0A7X5J8J1","uniprot"
"17245","CHEBI:17245","chebi","A0ABY6X4K8_9ENTR","A0ABY6X4K8","uniprot"
"17245","CHEBI:17245","chebi","A0ABW8Y7T1_9FLAO","A0ABW8Y7T1","uniprot"
"17245","CHEBI:1

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7136    0  7136    0     0   4895      0 --:--:--  0:00:01 --:--:--  4897


And the output on our terminal should be an even longer list of proteins related to carbon monoxide (ChEBI:17245) should be displayed.

The next step will show how to manage these long lists.

## Step 3: Output Redirection and File Management

If we want to analyze all the lines we can redirect the output to the command line tool less, which allows us to navigate through the output by using the arrow keys. To do that we can add the bar character (`|`) between two commands, which will transfer the output of the first command as input of the second. In a terminal, we would typically use `| less` to scroll through long outputs. However, in a notebook environment, using `less` does not work well because it requires interactive input. Instead, we can use the `head` command to preview just the first few lines of the output. The `-n 10` option tells `head` to show only the first 10 lines.

In [39]:
%%bash
./getproteins.sh 27732 | head -n 10

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"27732","CHEBI:27732","chebi","ICS1_CAMIR","Q2HXL9","uniprot"
"27732","CHEBI:27732","chebi","CKCS_CAMSB","A0A6C0WW38","uniprot"
"27732","CHEBI:27732","chebi","TCS1D_CAMTA","A0A0S2PMA8","uniprot"
"27732","CHEBI:27732","chebi","CHIB1_ASPFM","Q873X9","uniprot"
"27732","CHEBI:27732","chebi","RYR1_PIG","P16960","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFAR","Q9AVK1","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFCA","A0A096VHZ1","uniprot"
"27732","CHEBI:27732","chebi","TCS2_CAMSI","Q68CM3","uniprot"
"27732","CHEBI:27732","chebi","TCS3_CAMSI","A0A0S2PM82","uniprot"


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1890    0  1890    0     0   3197      0 --:--:-- --:--:-- --:--:--  3197


**Expected Output:** The first 10 lines of the protein list are displayed.

However, what we really want is to save the output as a file, not just
printing some characters on the screen. Thus, what we should do is redirect
the output to a CSV file. This can be done by adding the redirect operator `>`
and the filename, as described previously:

In [40]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1890    0  1890    0     0   3926      0 --:--:-- --:--:-- --:--:--  3929


**Expected Output:** No output displayed (file is being created/written).

To check if the file was really created and to analyze its contents, we can
use the `cat` command:

In [41]:
%%bash
cat chebi_27732_xrefs_UniProt.csv | head -n 10

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"27732","CHEBI:27732","chebi","ICS1_CAMIR","Q2HXL9","uniprot"
"27732","CHEBI:27732","chebi","CKCS_CAMSB","A0A6C0WW38","uniprot"
"27732","CHEBI:27732","chebi","TCS1D_CAMTA","A0A0S2PMA8","uniprot"
"27732","CHEBI:27732","chebi","CHIB1_ASPFM","Q873X9","uniprot"
"27732","CHEBI:27732","chebi","RYR1_PIG","P16960","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFAR","Q9AVK1","uniprot"
"27732","CHEBI:27732","chebi","CS3_COFCA","A0A096VHZ1","uniprot"
"27732","CHEBI:27732","chebi","TCS2_CAMSI","Q68CM3","uniprot"
"27732","CHEBI:27732","chebi","TCS3_CAMSI","A0A0S2PM82","uniprot"


The first ten lines of the `chebi_27732_xrefs_UniProt.csv` file are displayed.

We can also open the file in our spreadsheet application, such as LibreOffice Calc or Microsoft Excel.

## Conclusion

This concludes the **Biomedical Data Retrieval** tutorial adapted from the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to retrieve data from biomedical databases using `curl` and web services.

The next tutorial in this series will explore data extraction techniques to filter out irrelevant data from a CSV file, focusing on our specific information needs.

## Exercise 1

As an exercise execute the script to get the CSV file with the associated
proteins of [water](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15377) and [gold](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:30050).

In [42]:
%%bash
# Exercise: Get proteins for water (CHEBI:15377)
./getproteins.sh 15377 > water_proteins.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  7144    0  7144    0     0   7945      0 --:--:-- --:--:-- --:--:--  7937


**Expected Output:** No output displayed (file water_proteins.csv is being created).

In [43]:
%%bash
# Exercise: Get proteins for gold (CHEBI:30050)
./getproteins.sh 30050 > gold_proteins.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


**Expected Output:** No output displayed (file gold_proteins.csv is being created).

You can verify the files were created by listing them:

In [44]:
%%bash
ls -la *.csv

-rw-r--r-- 1 root root 1890 Aug 26 10:09 chebi_27732_xrefs_UniProt.csv
-rw-r--r-- 1 root root    0 Aug 26 10:09 gold_proteins.csv
-rw-r--r-- 1 root root 7144 Aug 26 10:09 water_proteins.csv
